# 03 — Results Analysis

Post-training analysis of the best checkpoint:
- Test set metrics (accuracy, top-5, macro F1)
- Confusion matrix
- Per-class F1 bar chart
- Grad-CAM gallery
- Error analysis (most confident wrong predictions)

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from dermnet.config import load_config
from dermnet.model import build_model
from dermnet.dataset import DermNetDataset, build_dataloaders
from dermnet.transforms import get_val_transforms
from dermnet.evaluate import run_evaluation
from dermnet.gradcam import visualize_gradcam_grid, get_target_layer

cfg = load_config('../configs/default.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
proc = Path('../data/processed')
ckpt_path = Path('../outputs/checkpoints/best.pt')

if not ckpt_path.exists():
    print('No checkpoint found. Run: make train  first.')
else:
    class_names = pd.read_csv(proc / 'classes.csv', index_col=0)['class_name'].tolist()
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=True)
    model = build_model(cfg.model.backbone, cfg.data.num_classes, pretrained=False,
                        dropout=cfg.model.dropout)
    model.load_state_dict(ckpt['model_state'])
    model.to(device).eval()
    print(f'Loaded checkpoint from epoch {ckpt["epoch"]}')

## 1. Test Set Metrics

In [ ]:
test_df = pd.read_csv(proc / 'test.csv')
val_df  = pd.read_csv(proc / 'val.csv')
train_df = pd.read_csv(proc / 'train.csv')

_, _, test_loader = build_dataloaders(
    train_df, val_df, test_df,
    get_val_transforms(cfg.data.image_size),
    get_val_transforms(cfg.data.image_size),
    batch_size=cfg.training.batch_size,
    num_workers=0,
    num_classes=cfg.data.num_classes
)

metrics = run_evaluation(model, test_loader, device, class_names,
                         figure_dir='../outputs/figures')

print('=' * 55)
print('TEST SET RESULTS')
print(f"  Accuracy    : {metrics['accuracy']:.4f}  "
      f"({metrics['accuracy']*100:.1f}%)")
print(f"  Top-5 Acc   : {metrics['top5_accuracy']:.4f}  "
      f"({metrics['top5_accuracy']*100:.1f}%)")
print(f"  Macro F1    : {metrics['macro_f1']:.4f}")
print(f"  Weighted F1 : {metrics['weighted_f1']:.4f}")
print('=' * 55)

## 2. Full Classification Report

In [ ]:
print(metrics['classification_report'])

## 3. Confusion Matrix

In [ ]:
from IPython.display import Image
Image('../outputs/figures/confusion_matrix.png')

## 4. Per-Class F1

In [ ]:
from IPython.display import Image
Image('../outputs/figures/per_class_f1.png')

## 5. Grad-CAM Gallery

Visualises WHERE the model looks when making predictions.
Red/warm regions = highest influence on the prediction.

In [ ]:
test_ds = DermNetDataset(test_df, get_val_transforms(cfg.data.image_size))
target_layer = get_target_layer(model)

visualize_gradcam_grid(
    model, target_layer, test_ds, class_names, device,
    num_images=8,
    save_path=Path('../outputs/figures/gradcam_grid.png')
)

from IPython.display import Image
Image('../outputs/figures/gradcam_grid.png')

## 6. Error Analysis — Most Confident Wrong Predictions

In [ ]:
# Re-run inference to collect per-sample details
records = []
model.eval()
with torch.no_grad():
    for i in range(len(test_ds)):
        tensor, true_label = test_ds[i]
        logits = model(tensor.unsqueeze(0).to(device))
        probs = torch.softmax(logits, dim=1).squeeze()
        pred_label = probs.argmax().item()
        confidence = probs[pred_label].item()
        if pred_label != true_label:
            records.append({
                'idx': i,
                'true': class_names[true_label],
                'pred': class_names[pred_label],
                'confidence': confidence,
                'path': test_df.iloc[i]['path'],
            })

errors_df = pd.DataFrame(records).sort_values('confidence', ascending=False)
print(f'Total errors: {len(errors_df)} / {len(test_ds)}')
print(f'Error rate: {len(errors_df)/len(test_ds)*100:.1f}%')
print('\nTop-10 most confident wrong predictions:')
print(errors_df[['true', 'pred', 'confidence']].head(10).to_string(index=False))

In [ ]:
# Visualise top-5 confident errors
import cv2
top_errors = errors_df.head(5)
fig, axes = plt.subplots(1, len(top_errors), figsize=(15, 4))
for ax, (_, row) in zip(axes, top_errors.iterrows()):
    img = cv2.cvtColor(cv2.imread(row['path']), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.axis('off')
    true_s = row['true'][:20]
    pred_s = row['pred'][:20]
    ax.set_title(f'True: {true_s}\nPred: {pred_s}\nConf: {row["confidence"]*100:.1f}%',
                 fontsize=7, color='red')
plt.suptitle('Most Confident Wrong Predictions', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/figures/top_errors.png', dpi=150)
plt.show()

## 7. Training History Curves

Loss and accuracy curves saved during training.

In [ ]:
from IPython.display import Image, display
for fig_name in ['loss_curves.png', 'accuracy_curves.png']:
    fig_path = f'../outputs/figures/{fig_name}'
    if Path(fig_path).exists():
        print(fig_name)
        display(Image(fig_path))